# Instalando dependências

In [ ]:
%pip install -r dependencias/requirements.txt

# Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Coleta de dados

In [ ]:
cestas = pd.read_csv("bases/cestas.csv")
clientes = pd.read_csv('bases/clientes.csv')
produtos = pd.read_csv('bases/produtos.csv')

# Produtos mais vendidos

In [ ]:
faturamento_por_produto = cestas.groupby('item_id').agg({'price': 'sum'}).reset_index()
top3_faturamento = faturamento_por_produto.sort_values('price', ascending=False).head(3)
top3_faturamento['rec_source'] = 'top products'

In [ ]:
top3_faturamento

# Definição de funções

In [ ]:
def prepare_data_apriori(df):

    transactions = df.groupby('basket_id')['item_id'].apply(list).tolist()
    
    encoder = TransactionEncoder()
    encoder_fitted = encoder.fit(transactions).transform(transactions)
    encoded_df = pd.DataFrame(encoder_fitted, columns = encoder.columns_)

    return encoded_df

In [ ]:
def compute_association_rules(encoded_df, min_support=0.01, top=20):

    apriori_associations = apriori(
        encoded_df,
        min_support=min_support,
        use_colnames=True
    )

    if len(apriori_associations) == 0:
        return pd.DataFrame()

    rules = association_rules(
        apriori_associations,
        metric="lift",
        min_threshold=1
    )

    if len(rules) == 0:
        return pd.DataFrame()

    rules = rules.drop_duplicates(subset=['antecedents', 'consequents'])

    rules = rules.sort_values(
        by=["lift", "confidence", "support"],
        ascending=False
    ).head(top)

    return rules

In [ ]:
def recommend_for_users(df_baskets, rules):

    df_baskets["timestamp"] = pd.to_datetime(df_baskets["timestamp"], errors="coerce")

    df_mes = df_baskets[
        (df_baskets["timestamp"].dt.year == 2025) &
        (df_baskets["timestamp"].dt.month == 12)
    ].copy()

    recomendacoes = []

    for user_id, group in df_mes.groupby("user_id"):

        itens_user = set(group["item_id"].unique())

        melhores_por_consequente = {}

        for _, rule in rules.iterrows():

            antecedent = set(rule["antecedents"])
            consequent = set(rule["consequents"])

            if not antecedent.issubset(itens_user):
                continue

            novos_itens = list(consequent - itens_user)

            if len(novos_itens) == 0:
                continue

            for item_rec in novos_itens:

                if item_rec in melhores_por_consequente:
                    if rule["lift"] > melhores_por_consequente[item_rec]["lift"]:
                        melhores_por_consequente[item_rec] = {
                            "user_id": user_id,
                            "antecedent": tuple(antecedent),
                            "consequent": item_rec,
                            "support": rule["support"],
                            "confidence": rule["confidence"],
                            "lift": rule["lift"]
                        }
                else:
                    melhores_por_consequente[item_rec] = {
                        "user_id": user_id,
                        "antecedent": tuple(antecedent),
                        "consequent": item_rec,
                        "support": rule["support"],
                        "confidence": rule["confidence"],
                        "lift": rule["lift"]
                    }

        recomendacoes.extend(melhores_por_consequente.values())

    df_rec = pd.DataFrame(recomendacoes)

    if len(df_rec) > 0:
        df_rec = df_rec.sort_values(
            by=["user_id", "lift", "confidence", "support"],
            ascending=[True, False, False, False]
        )

    return df_rec

In [ ]:
def completar_recomendacoes(grupo):
    grupo = grupo.copy()

    recomendações_validas = grupo[grupo['rec'].notna()].head(3)
    n_validas = len(recomendações_validas)
    
    if n_validas >= 3:
        return recomendações_validas.head(3)

    linhas_necessarias = 3 - n_validas
    produtos_top = top3_faturamento.head(linhas_necessarias).copy()

    for _, produto in produtos_top.iterrows():
        nova_linha = pd.DataFrame({
            'user_id': [grupo['user_id'].iloc[0]],
            'rec': [produto['item_id']],
            'source_rec': ['top products']
        })
        recomendações_validas = pd.concat([recomendações_validas, nova_linha], ignore_index=True)
    
    return recomendações_validas

In [ ]:
encoded_df_cestas = prepare_data_apriori(cestas)
rules = compute_association_rules(encoded_df_cestas, min_support=0.01, top=30)

In [ ]:
rules.head()

In [ ]:
recs = recommend_for_users(cestas, rules)

In [ ]:
recs.head()

In [ ]:
df_merged = pd.merge(recs, clientes, on='user_id', how='outer')
df_merged['source_rec'] = np.where(df_merged['consequent'].notna(), 'apriori', np.nan)
df_merged = df_merged[['user_id', 'consequent', 'source_rec']].rename(columns={'consequent': 'rec'})

In [ ]:
df_merged

In [ ]:
df_recs_preenchido = df_merged.groupby('user_id').apply(completar_recomendacoes).reset_index(drop=True)

In [ ]:
df_recs_preenchido

In [ ]:
df_completo = pd.merge(df_recs_preenchido, clientes, on='user_id', how='left')
df_completo = pd.merge(df_completo, produtos, left_on='rec', right_on='item_id', how='left')

In [ ]:
df_output = df_completo[['user_id','nome','documento','telefone','uf','cluster','item_id','item_desc','source_rec']].rename(columns={'item_desc': 'rec_desc', 'item_id': 'rec_id'})

In [ ]:
df_output.head()

In [ ]:
df_output.to_csv('recomendacoes.csv', index=False)